🟩 Cell 1 — Import Library

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

🟩 Cell 2 — Parameter Konfigurasi

In [ ]:
ASPECT = 'price'

TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO  = 0.15

TOLERANCE = 0.05
MAX_SEED_TRIES = 200

INPUT_PATH = 'data_clean/dataset_absa_clean.csv'
OUTPUT_DIR = 'data_split'

os.makedirs(OUTPUT_DIR, exist_ok=True)

🟩 Cell 3 — Load Dataset & Filter Per Aspek

In [3]:
df = pd.read_csv(INPUT_PATH, sep=';')

print(f"Total rows (all aspects): {len(df)}")

valid_labels = ['positive', 'negative', 'neutral']

if ASPECT not in df.columns:
    raise ValueError(f"Aspek {ASPECT} tidak ditemukan di dataset")

df_aspect = df[df[ASPECT].isin(valid_labels)].copy()

print(f"\n=== ASPECT: {ASPECT} ===")
print(f"Total rows untuk aspek {ASPECT}: {len(df_aspect)}")

global_counts = df_aspect[ASPECT].value_counts()
global_props  = df_aspect[ASPECT].value_counts(normalize=True)

print("\nDistribusi label GLOBAL:")
print(pd.concat([global_counts, global_props.rename("proportion")], axis=1))

Total rows (all aspects): 9194

=== ASPECT: price ===
Total rows untuk aspek price: 9190

Distribusi label GLOBAL:
          count  proportion
price                      
neutral    6676    0.726442
positive   2318    0.252231
negative    196    0.021328


🟩 Cell 4 — Fungsi Cek Toleransi Distribusi

In [4]:
def within_tolerance(global_props, split_props, tolerance=0.05):
    labels = set(global_props.index) | set(split_props.index)
    for lbl in labels:
        g = global_props.get(lbl, 0.0)
        s = split_props.get(lbl, 0.0)
        if abs(s - g) > tolerance:
            return False
    return True

🟩 Cell 5 — Stratified Split 70/15/15 + Toleransi 5%

In [5]:
best_seed = None
best_split = None

for seed in range(42, 42 + MAX_SEED_TRIES):
    train_df, temp_df = train_test_split(
        df_aspect,
        train_size=TRAIN_RATIO,
        stratify=df_aspect[ASPECT],
        random_state=seed
    )

    valid_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        stratify=temp_df[ASPECT],
        random_state=seed
    )

    if (
        within_tolerance(global_props, train_df[ASPECT].value_counts(normalize=True), TOLERANCE)
        and within_tolerance(global_props, valid_df[ASPECT].value_counts(normalize=True), TOLERANCE)
        and within_tolerance(global_props, test_df[ASPECT].value_counts(normalize=True), TOLERANCE)
    ):
        best_seed = seed
        best_split = (train_df, valid_df, test_df)
        break

if best_split is None:
    print("⚠️ Tidak menemukan seed ideal, gunakan stratified default (seed=42)")
    train_df, temp_df = train_test_split(
        df_aspect,
        train_size=TRAIN_RATIO,
        stratify=df_aspect[ASPECT],
        random_state=42
    )
    valid_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        stratify=temp_df[ASPECT],
        random_state=42
    )
else:
    train_df, valid_df, test_df = best_split
    print(f"✅ Menggunakan seed terbaik: {best_seed}")

✅ Menggunakan seed terbaik: 42


🟩 Cell 6 — Cek Distribusi Setelah Split

In [6]:
def show_dist(name, df):
    print(f"\nDistribusi {name}:")
    print(df[ASPECT].value_counts(normalize=True))

show_dist("TRAIN", train_df)
show_dist("VALID", valid_df)
show_dist("TEST ", test_df)


Distribusi TRAIN:
price
neutral     0.726411
positive    0.252293
negative    0.021296
Name: proportion, dtype: float64

Distribusi VALID:
price
neutral     0.726415
positive    0.251814
negative    0.021771
Name: proportion, dtype: float64

Distribusi TEST :
price
neutral     0.726613
positive    0.252357
negative    0.021030
Name: proportion, dtype: float64


🟩 Cell 7 — Format Output (sentence, label)

In [7]:
cols = ['review', ASPECT]

train_out = train_df[cols].rename(columns={'review':'sentence', ASPECT:'label'})
valid_out = valid_df[cols].rename(columns={'review':'sentence', ASPECT:'label'})
test_out  = test_df[cols].rename(columns={'review':'sentence', ASPECT:'label'})

🟩 Cell 8 — 🔥 KHUSUS TEST: Ubah Semua Label Menjadi neutral

In [8]:
test_out_neutral = test_out.copy()
test_out_neutral['label'] = 'neutral'

print("\nContoh data TEST setelah label dinetralkan:")
test_out_neutral.head()


Contoh data TEST setelah label dinetralkan:


,sentence,label
6682,baru kemarin banget makan disini hampir menit ...,neutral
5846,tempatnya nyaman dan bersih pelayanannya sanga...,neutral
2686,ayam nya dikit daging nya juga enggak ada,neutral
2511,tempatnya oke harganya murah dan terjangkau ra...,neutral
9034,makanan khas banyuwangi sambelnya enak,neutral


🟩 Cell 9 — Simpan ke CSV

In [9]:
train_path = f"{OUTPUT_DIR}/{ASPECT}_data_train.csv"
valid_path = f"{OUTPUT_DIR}/{ASPECT}_data_valid.csv"
test_path  = f"{OUTPUT_DIR}/{ASPECT}_data_test_neutral.csv"

train_out.to_csv(train_path, index=False)
valid_out.to_csv(valid_path, index=False)
test_out_neutral.to_csv(test_path, index=False)

print("\n✅ File berhasil disimpan:")
print(train_path)
print(valid_path)
print(test_path)



✅ File berhasil disimpan:
data_split/price_data_train.csv
data_split/price_data_valid.csv
data_split/price_data_test_neutral.csv
